# Superstore SQL Analysis
## Business questions answered using SQL queries on the dataset

In [2]:
import sqlite3
import pandas as pd

#loading csv file
df = pd.read_csv('sample_-_superstore.csv')

#fixing date's data types
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

#adding year & month
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month

#creating database
conn = sqlite3.connect('superstore.db')
df.to_sql('orders', conn, if_exists='replace', index=False)

print("database ready")

database ready


## 1. Regional Sales and Profit Analysis

In [3]:
query = """
SELECT Region, 
       ROUND(SUM(Sales), 2) as Total_Sales,
       ROUND(SUM(Profit), 2) as Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales) * 100, 2) as Profit_Margin
FROM orders
GROUP BY Region
ORDER BY Total_Profit DESC
"""

pd.read_sql_query(query, conn)
    

,Region,Total_Sales,Total_Profit,Profit_Margin
0,West,739813.61,110798.82,14.98
1,East,691828.17,94883.26,13.71
2,South,391721.91,46749.43,11.93
3,Central,503170.67,39865.31,7.92


### Finding
West leads in both sales and profit margin at 14.98%.
Central has the worst Profit margin at 7.92% despite being 3rd in sales.
Same finding as Python analysis — confirmed.

## 2. Category Performance by Region

In [4]:
query = """
SELECT Region,
       Category,
       ROUND(SUM(Sales),2) as Total_Sales,
       ROUND(SUM(Profit),2) as Total_Profit,
       ROUND(AVG(Discount) * 100, 2) as Avg_Discount_Pct,
       ROUND(SUM(Profit)/SUM(Sales) * 100 ,2) as Profit_margin
FROM Orders
GROUP BY Region, Category
ORDER BY Region, Total_Profit DESC
"""

pd.read_sql_query(query, conn)

,Region,Category,Total_Sales,Total_Profit,Avg_Discount_Pct,Profit_margin
0,Central,Technology,170416.31,33697.43,13.31,19.77
1,Central,Office Supplies,168216.71,8970.08,25.36,5.33
2,Central,Furniture,164537.65,-2802.21,29.74,-1.70
3,East,Technology,267938.07,48441.78,14.07,18.08
4,East,Office Supplies,211658.40,42996.74,14.03,20.31
5,East,Furniture,212231.70,3444.74,15.44,1.62
6,South,Technology,148771.91,19991.83,10.78,13.44
7,South,Office Supplies,125651.31,19986.39,16.74,15.91
8,South,Furniture,117298.68,6771.21,12.15,5.77
9,West,Office Supplies,226366.89,54070.23,9.29,23.89


### Finding
1. Central Furniture average discount is 29.74% — highest in the entire 
dataset — resulting in a net loss of 2,802.
Even South Furniture with only 12% discount barely makes 5.7% margin 
on $117k sales — confirming Furniture is structurally Costly for business
regardless of discounting.

2. Technology is the most consistent performer across all regions 
with margins between 13-19%.

3. East Office Supplies — 14.03% discount → 20.31% margin, 
Central Office Supplies — 25.36% discount → 5.33% margin
same category but central is giving 11% more discount and getting 15% less margin.

4. West Office Supplies — only 9.29% discount → 23.89% margin — highest margin in the entire table.
Office Supplies is actually a great category when discounts are controlled. Central is destroying it with poor discount management.




## 3. Yearly Business Trend

In [5]:
query = """
SELECT Year,
       ROUND(SUM(Sales),2) as Total_Sales,
       ROUND(SUM(Profit),2) as Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales) * 100,2) as Profit_Margin

FROM orders
GROUP BY Year
ORDER BY Year
"""

pd.read_sql_query(query, conn)

,Year,Total_Sales,Total_Profit,Profit_Margin
0,2023,494040.21,51684.30,10.46
1,2024,472993.03,62020.97,13.11
2,2025,613933.58,82665.20,13.46
3,2026,745567.53,95926.35,12.87


### Finding
2024 shows an interesting anomaly — sales declined slightly but 
profit margin improved to 13.11%. The business sold less but smarter.

From 2025 onwards revenue grew strongly but margin remained flat 
between 13-13.5% — suggesting the business is scaling volume 
without fixing its underlying discount and profitability problems.

Revenue grew 50% from 2023 to 2026 but margin only improved 2.4% — 
the business is getting bigger but not more efficient.

2026 shows highest ever revenue at $745k but profit margin 
has started declining to 12.87% — below both 2024 and 2025 levels.
If this trend continues the business risks growing revenue while 
shrinking profitability — a dangerous pattern for long term health.

## 4. Customer Segment Analysis

In [6]:
query = """
SELECT Segment,
       ROUND(SUM(Sales), 2) as Total_Sales,
       ROUND(SUM(Profit), 2) as Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales) * 100, 2) as Profit_Margin,
       ROUND(AVG(Discount) * 100, 2) as Avg_Discount_Pct
FROM orders
GROUP BY Segment
ORDER BY Profit_Margin DESC
"""

pd.read_sql_query(query, conn)

,Segment,Total_Sales,Total_Profit,Profit_Margin,Avg_Discount_Pct
0,Home Office,440068.43,61675.73,14.02,14.71
1,Corporate,715806.13,94249.64,13.17,15.65
2,Consumer,1170659.79,136371.45,11.65,15.76


### Finding
Home Office is the smallest segment by revenue but most profitable 
by margin at 14.02% — receiving the lowest average discount at 14.71%.

Consumer drives the highest revenue at $1.17M but has the lowest 
margin at 11.65% — highest discount at 15.76% is eating into profits.

Pattern confirmed: lower discounts = higher margins across all segments.

## 5.Shipping Impact on business

In [7]:
query = """
SELECT [Ship Mode],
       COUNT(DISTINCT [Order ID]) as Total_Orders,
       ROUND(SUM(Sales), 2) as Total_Sales,
       ROUND(SUM(Profit), 2) as Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales) * 100, 2) as Profit_Margin
FROM orders
GROUP BY [Ship Mode]
ORDER BY Profit_Margin DESC
"""

pd.read_sql_query(query, conn)


,Ship Mode,Total_Orders,Total_Sales,Total_Profit,Profit_Margin
0,First Class,795,351750.74,49012.72,13.93
1,Second Class,982,466671.11,58962.13,12.63
2,Same Day,266,129271.96,16160.75,12.50
3,Standard Class,3068,1378840.55,168161.21,12.20


### Region:

In [8]:
query = """
SELECT Region,
       [Ship Mode],
       COUNT(DISTINCT [Order ID]) as Total_Orders,
       ROUND(SUM(Sales), 2) as Total_Sales,
       ROUND(SUM(Profit), 2) as Total_Profit,
       ROUND(SUM(Profit)/SUM(Sales) * 100, 2) as Profit_Margin
FROM orders
GROUP BY Region, [Ship Mode]
ORDER BY Region, Profit_Margin DESC
"""

pd.read_sql_query(query, conn)

,Region,Ship Mode,Total_Orders,Total_Sales,Total_Profit,Profit_Margin
0,Central,Second Class,224,103550.01,9114.83,8.80
1,Central,Standard Class,717,320291.18,25533.27,7.97
2,Central,Same Day,62,20415.41,1531.88,7.50
3,Central,First Class,176,58914.08,3685.33,6.26
4,East,Same Day,76,44235.66,8249.26,18.65
5,East,Standard Class,884,415028.52,59409.91,14.31
6,East,First Class,239,113742.20,15796.84,13.89
7,East,Second Class,276,118821.79,11427.26,9.62
8,South,Second Class,164,93758.61,14667.15,15.64
9,South,First Class,124,49332.57,6892.39,13.97


### Finding
Central has no standout shipping mode — ALL modes have poor margins 
between 6-8%, confirming the problem is not shipping related but 
discount and product mix related.

South Same Day is an anomaly — negative margin at -8.39%. 
Likely shipping costs exceed revenue on urgent deliveries in this region.

West and East perform well across all shipping modes — 
suggesting regional management quality plays a bigger role 
than shipping method choice.

In [9]:
query = """
SELECT [State/Province],
       Region,
       ROUND(SUM(Sales), 2) as Total_Sales,
       ROUND(SUM(Profit), 2) as Total_Profit,
       ROUND(AVG(Discount),2) as Disc_pct,
       ROUND(SUM(Profit)/SUM(Sales) * 100, 2) as Profit_Margin
FROM orders
GROUP BY [State/Province], Region
ORDER BY Total_Profit ASC
LIMIT 15
"""

pd.read_sql_query(query, conn)

,State/Province,Region,Total_Sales,Total_Profit,Disc_pct,Profit_Margin
0,Texas,Central,170188.05,-25729.36,0.37,-15.12
1,Ohio,East,78258.14,-16971.38,0.32,-21.69
2,Pennsylvania,East,116511.91,-15559.96,0.33,-13.35
3,Illinois,Central,80166.10,-12607.89,0.39,-15.73
4,North Carolina,South,55603.16,-7490.91,0.28,-13.47
5,Colorado,West,32108.12,-6527.86,0.32,-20.33
6,Tennessee,South,30661.87,-5341.69,0.29,-17.42
7,Arizona,West,35282.00,-3427.92,0.30,-9.72
8,Florida,South,89473.71,-3399.30,0.30,-3.80
9,Oregon,West,17431.15,-1190.47,0.29,-6.83


In [10]:
query = """
SELECT [State/Province],
       Region,
       [Ship Mode],
       COUNT(DISTINCT [Order ID]) as Total_Orders,
       ROUND(SUM(Sales), 2) as Total_Sales,
       ROUND(SUM(Profit), 2) as Total_Profit,
       ROUND(AVG(Discount) * 100, 2) as Avg_Discount_Pct,
       ROUND(SUM(Profit)/SUM(Sales) * 100, 2) as Profit_Margin
FROM orders
GROUP BY [State/Province], Region, [Ship Mode]
ORDER BY Total_Profit ASC
LIMIT 15
"""

pd.read_sql_query(query, conn)

,State/Province,Region,Ship Mode,Total_Orders,Total_Sales,Total_Profit,Avg_Discount_Pct,Profit_Margin
0,Texas,Central,Standard Class,286,104754.88,-19592.65,38.09,-18.70
1,Ohio,East,Standard Class,119,39944.62,-11016.08,33.09,-27.58
2,Illinois,Central,Standard Class,173,48403.95,-8981.64,39.66,-18.56
3,Pennsylvania,East,Standard Class,175,69164.27,-8715.72,33.15,-12.60
4,Colorado,West,Standard Class,41,21627.63,-5221.60,31.72,-24.14
5,Pennsylvania,East,Second Class,48,26284.55,-4854.49,32.86,-18.47
6,Florida,South,Standard Class,125,62491.44,-4187.38,31.50,-6.70
7,North Carolina,South,Same Day,11,12257.10,-3666.03,27.14,-29.91
8,Ohio,East,First Class,47,15615.94,-3494.84,34.41,-22.38
9,Tennessee,South,Standard Class,65,21300.67,-2961.01,28.51,-13.90


In [11]:
query = """
SELECT [State/Province],
       Region,
       [Ship Mode],
       COUNT(DISTINCT [Order ID]) as Total_Orders,
       ROUND(SUM(Sales), 2) as Total_Sales,
       ROUND(SUM(Profit), 2) as Total_Profit,
       ROUND(AVG(Discount) * 100, 2) as Avg_Discount_Pct,
       ROUND(SUM(Profit)/SUM(Sales) * 100, 2) as Profit_Margin
FROM orders
GROUP BY [State/Province], Region, [Ship Mode]
ORDER BY Total_Profit DESC
LIMIT 15
"""

pd.read_sql_query(query, conn)

,State/Province,Region,Ship Mode,Total_Orders,Total_Sales,Total_Profit,Avg_Discount_Pct,Profit_Margin
0,California,West,Standard Class,601,255739.17,43626.00,7.23,17.06
1,New York,East,Standard Class,344,181606.59,42046.30,5.87,23.15
2,New York,East,First Class,80,60972.79,16755.25,4.80,27.48
3,Washington,West,Standard Class,155,68517.08,14859.23,6.53,21.69
4,California,West,Second Class,201,92692.98,14762.88,7.32,15.93
5,Michigan,Central,Standard Class,72,44045.82,14351.84,0.65,32.58
6,Indiana,Central,Standard Class,45,39318.26,14155.78,0.00,36.00
7,California,West,First Class,164,79476.25,12397.89,7.53,15.60
8,Washington,West,First Class,31,31732.60,11620.02,6.18,36.62
9,Georgia,South,Standard Class,56,30833.13,10944.16,0.00,35.49


### The Complete Picture — Proof

Top performing state-shipping combinations have near zero discounts:
- Indiana Standard Class: 0% discount → 36% margin
- Georgia Standard Class: 0% discount → 35.49% margin  
- Minnesota Standard Class: 0% discount → 37.55% margin

Compare to worst performing:
- Texas Standard Class: 38% discount → -18.70% margin
- Ohio Standard Class: 33% discount → -27.58% margin

Standard Class appears in BOTH best and worst combinations —
proving shipping mode alone does not determine profitability.

The difference between a 36% profit margin and a -18% loss 
is entirely explained by discount policy — not shipping method.

## Discount's relation to profit

In [12]:
query = """
SELECT 
    CASE 
        WHEN Discount = 0 THEN '0% No Discount'
        WHEN Discount <= 0.10 THEN '1-10% Low'
        WHEN Discount <= 0.20 THEN '11-20% Moderate'
        WHEN Discount <= 0.30 THEN '21-30% High'
        ELSE '30%+ Dangerous'
    END as Discount_Band,
    COUNT(DISTINCT [Order ID]) as Total_Orders,
    ROUND(AVG(Profit), 2) as Avg_Profit,
    ROUND(SUM(Profit)/SUM(Sales) * 100, 2) as Profit_Margin
FROM orders
GROUP BY Discount_Band
ORDER BY Profit_Margin DESC
"""

pd.read_sql_query(query, conn)

,Discount_Band,Total_Orders,Avg_Profit,Profit_Margin
0,0% No Discount,2714,66.34,29.56
1,1-10% Low,91,94.79,16.56
2,11-20% Moderate,2469,24.61,11.54
3,21-30% High,214,-45.71,-10.06
4,30%+ Dangerous,904,-105.91,-48.22


### Insight:
The business crosses into loss territory the moment 
discount exceeds 20%. Above 30% the business loses 
nearly half of every dollar of revenue.

904 orders have 30%+ discounts — these alone are 
destroying company profitability.

Core recommendation: Hard cap discounts at 20% company wide.
No exceptions. No state, region or category justifies 
going beyond this threshold based on this data.